<!-- nav -->
[← 8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb) · [Home](../../README.md) · [8.15 · SystemVerilog Assertions →](08_15_SystemVerilog_Assertions.ipynb)

# 8.14 · SystemVerilog Patterns

Tutorial 8 showed a `matches` comparing a value with a pattern, `v matches '{.a, .b}`, and binding the pattern's
variables (08.2). This page takes the patterns themselves, IEEE 1800's pattern matching (12.6), as their dialect,
`SystemVerilog Patterns`, models them: from the smallest, a variable and a wildcard, to tagged and structure patterns,
each in the builders' fluent calls beside the text SystemVerilog writes. Then what is done with them: what each binds;
validated, with one refused; held by a `matches`, which binds them in a conditional's consequence; and never simplified, each kind left in the residual, named. Last, the editions that have them.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4); `variable` writes a variable
pattern, `.x`, of its spelling, as tutorial 5's `variable` writes a name; `binds` lists the names a pattern binds, and
`refused` gives what a simplifier says of an expression, its value where its residual is a literal, or why it has none, as tutorial 8's
does.

In [1]:
import assert from "node:assert/strict";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import { Domains, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Patterns = SVPatterns.Expressions.DIALECT; // the dialect, as the framework reads it
const name = (spelling: string) => (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const literal = (value: unknown) => (d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(value)));
const variable = (spelling: string) => (p: any) => p.variablepattern((v: any) => v.name((i: any) => i.spelling(spelling))); // .spelling

/** The names a pattern binds, in order. */
function binds(expression: unknown): string {
  return [...SVPatterns.Expressions.boundNames(Terms.term(expression))].sort().join(", ") || "none";
}

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(simplify: (expression: any) => unknown, expression: any): string {
  try {
    const result: any = simplify(expression);
    return String(typeof result?.evaluate === "function" ? result.evaluate() : result);
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.14.1"></a>
## 08.14.1 · A variable and a wildcard

```text
.x        .*
```

The smallest patterns match anything. `.x` is a variable pattern, `variablepattern`: it matches any value and binds
the name `x` to it, its name `Expressions`' identifier helper, the one a name expression holds (tutorial 4). `.*` is a
wildcard, `wildcardpattern`, which matches any value and binds nothing. Each is a term of `SystemVerilog Patterns`, the
root of its holder, `S.SystemVerilog.Patterns.Expression`, and the family's printer writes it:

In [2]:
const smallest: any[] = store.transaction(() => [variable("x"), (p: any) => p.wildcardpattern((w: any) => w)]
  .map((root) => S.SystemVerilog.Patterns.Expression((e: any) => e.root(root))));
for (const pattern of smallest) {
  const form = Terms.form(Terms.term(pattern));
  console.log(SystemVerilog.Text.ToText(pattern).padEnd(3), form.kind.padEnd(16), "binds", binds(pattern).padEnd(5), Terms.dialectOf(Terms.term(pattern)));
}
assert.deepEqual(smallest.map((p) => SystemVerilog.Text.ToText(p)), [".x", ".*"]);

.x  variablepattern  binds x     Expressions.SystemVerilog.Patterns.Term


.*  wildcardpattern  binds none  Expressions.SystemVerilog.Patterns.Term


<a id="T08.14.2"></a>
## 08.14.2 · Tagged, structure and parenthesized patterns

```text
tagged Valid .x        tagged None
'{.a, .b}              '{a: .x, b: 1}
(tagged Valid .x)
```

A tagged union's value is matched by its member: `tagged Valid .x` matches the union when `Valid` is its member, and
matches the member's value with `.x`; `tagged None` names a member with no value. A structure is matched member by
member, in order, `'{.a, .b}`, or by name, `'{a: .x, b: 1}`, each named member a `patternmember`. Where a pattern is an
expression, `1` here, it is a constant pattern, which matches a value equal to it: no kind of its own, but
`Expressions`' own literal, since the dialect's `Term` lists `Expressions`' branches (tutorial 8, 08.3). Parentheses
written around a pattern are kept, a `parenthesizedpattern`. Patterns nest, and what a pattern binds is every variable
within it:

In [3]:
const valid = (p: any) => p.taggedpattern((t: any) => t.member((i: any) => i.spelling("Valid")).pattern(variable("x")));
const nested: any[] = store.transaction(() => [
  valid,
  (p: any) => p.taggedpattern((t: any) => t.member((i: any) => i.spelling("None"))),
  (p: any) => p.structurepattern((s: any) => s.items([variable("a"), variable("b")])),
  (p: any) => p.structurepattern((s: any) => s.items([
    (m: any) => m.patternmember((p: any) => p.name((i: any) => i.spelling("a")).pattern(variable("x"))),
    (m: any) => m.patternmember((p: any) => p.name((i: any) => i.spelling("b")).pattern(literal(1n)))])),
  (p: any) => p.parenthesizedpattern((q: any) => q.pattern(valid)),
].map((root) => S.SystemVerilog.Patterns.Expression((e: any) => e.root(root))));
for (const pattern of nested) {
  const form = Terms.form(Terms.term(pattern));
  console.log(SystemVerilog.Text.ToText(pattern).padEnd(18), form.kind.padEnd(20), "binds", binds(pattern).padEnd(5), "| its terms:",
    form.arguments.filter((a: any) => a != null).map((a: any) => Terms.kindOf(a)[0]).join(", ") || "none");
}
assert.deepEqual(nested.map((p) => SystemVerilog.Text.ToText(p)), [
  "tagged Valid .x", "tagged None", "'{.a, .b}", "'{a: .x, b: 1}", "(tagged Valid .x)"]);

tagged Valid .x    taggedpattern        binds x     | its terms: variablepattern


tagged None        taggedpattern        binds none  | its terms: none


'{.a, .b}          structurepattern     binds a, b  | its terms: variablepattern, variablepattern


'{a: .x, b: 1}     structurepattern     binds x     | its terms: patternmember, patternmember


(tagged Valid .x)  parenthesizedpattern binds x     | its terms: taggedpattern


A pattern binds, but it is no binder of its own: none of its kinds refines the framework's `Let` (tutorial 7). It binds
through what holds it, a `matches` (08.14.4).

<a id="T08.14.3"></a>
## 08.14.3 · Validated, and one refused

`validate` checks a pattern as it checks any term: what each kind requires, its identifiers' spellings, and the names its
constant patterns read, which are references, not bindings, so the caller must bind them. A structure pattern's members
are all in order or all by name (IEEE 1800 12.6), so one that mixes them is refused:

In [4]:
const [constant, mixed, misspelled]: any[] = store.transaction(() => [
  (p: any) => p.structurepattern((s: any) => s.items([
    variable("a"), (e: any) => e.binaryexpression((o: any) => o.left(name("n")).operator("+").right(literal(1n)))])),
  (p: any) => p.structurepattern((s: any) => s.items([
    (m: any) => m.patternmember((p: any) => p.name((i: any) => i.spelling("a")).pattern(variable("x"))), variable("y")])),
  variable("1x"),
].map((root) => S.SystemVerilog.Patterns.Expression((e: any) => e.root(root))));
const checked: [any, string[]][] = [[constant, []], [constant, ["n"]], [mixed, []], [misspelled, []]];
for (const [pattern, bound] of checked) {
  console.log(SystemVerilog.Text.ToText(pattern).padEnd(13), (bound.join(", ") || "-").padEnd(3), Patterns.validate(pattern, bound).join("; ") || "valid");
}
assert.deepEqual(Patterns.validate(checked[2]![0]), ["a StructurePattern's members are in order or by name, not both"]);

'{.a, n + 1}  -   argument 1: left: nameexpression 'n' is not bound


'{.a, n + 1}  n   valid


'{a: .x, .y}  -   a StructurePattern's members are in order or by name, not both


.1x           -   '1x' is not an identifier


<a id="T08.14.4"></a>
## 08.14.4 · Held by a `matches`

```text
v matches tagged Valid .x ? x + 1 : 0
v matches '{.a, .b} &&& a > b ? a : b
```

A pattern stands in one place in SystemVerilog's expressions: a `matches`' pattern, the one property of `Expressions`'
kinds that holds this dialect's `Term`, read from the store. A `matches` binds its pattern's variables where its
conditional is true: in the consequence, and in the conditions after it in a `&&&` (12.6.2), never in the value it
matches or in the alternative. So `x` is bound in `x + 1`, and only `v` is free; in the second, `a > b` reads what the
pattern bound:

In [5]:
const union = store.registered(Patterns.union);
const host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches as any[];
console.log(host.flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, prop]: [string, any]) => prop.schema === union || prop.schema.element === union)
  .map(([p]: [string, any]) => `${b.name}.${p}`)).join(", "));
const chosen: any[] = store.transaction(() => [
  (u: any) => u.conditionalexpression((c: any) => c
    .condition((d: any) => d.matchesexpression((m: any) => m.value(name("v")).pattern(valid)))
    .consequence((d: any) => d.binaryexpression((b: any) => b.left(name("x")).operator("+").right(literal(1n))))
    .alternative(literal(0n))),
  (u: any) => u.conditionalexpression((c: any) => c
    .condition((d: any) => d.predicateexpression((p: any) => p.conditions([
      (d: any) => d.matchesexpression((m: any) => m.value(name("v")).pattern(
        (p: any) => p.structurepattern((s: any) => s.items([variable("a"), variable("b")])))),
      (d: any) => d.binaryexpression((b: any) => b.left(name("a")).operator(">").right(name("b")))])))
    .consequence(name("a")).alternative(name("b"))),
].map((root) => S.SystemVerilog.Expressions.Expression((e: any) => e.root(root))));
const dialect = SystemVerilog.Expressions.DIALECT;
for (const expression of chosen) {
  console.log(SystemVerilog.Text.ToText(expression).padEnd(38), "free:", [...Symbolics.free(expression, dialect)].sort().join(", "),
    "| problems:", dialect.validate(expression, ["v"]).join("; ") || "none");
}
assert.deepEqual(chosen.map((e) => SystemVerilog.Text.ToText(e)), [
  "v matches tagged Valid .x ? x + 1 : 0", "v matches '{.a, .b} &&& a > b ? a : b"]);

matchesexpression.pattern


v matches tagged Valid .x ? x + 1 : 0  free: v | problems: none


v matches '{.a, .b} &&& a > b ? a : b  free: b, v | problems: alternative: nameexpression 'b' is not bound


The second's alternative reads `b`, which the pattern did not bind there: the conditional's scopes say so, and
`validate`, given `v` alone, reports the alternative's `b` as unbound, in the words it reports any free name with.

<a id="T08.14.5"></a>
## 08.14.5 · Never simplified

SystemVerilog's rules match no pattern (stage 6, "Q11"): they fold none of the dialect's kinds: each stays in the residual, named, held alone
or by a `matches`, and a conditional sizes its condition first, so the `matches` stays, named by its own kind before its
consequence, which reads what the pattern binds, is read. A pattern has no domain to infer, `any`:

In [6]:
const evaluate = SystemVerilog.Simplifiers.OfAny;
for (const pattern of [...smallest, ...nested.slice(0, 3)]) {
  console.log(SystemVerilog.Text.ToText(pattern).padEnd(16), refused((e) => evaluate(e), pattern));
}
console.log(SystemVerilog.Text.ToText(chosen[0]).padEnd(16), refused((e) => evaluate(e, { v: 1n }), chosen[0]));
console.log("inferred:", Patterns.infer(nested[0]).name());
assert.ok(Patterns.infer(nested[0]) === Domains.Anything);

.x               SystemVerilog Patterns variablepattern is not evaluated


.*               SystemVerilog Patterns wildcardpattern is not evaluated


tagged Valid .x  SystemVerilog Patterns taggedpattern is not evaluated


tagged None      SystemVerilog Patterns taggedpattern is not evaluated


'{.a, .b}        SystemVerilog Patterns structurepattern is not evaluated


v matches tagged Valid .x ? x + 1 : 0 SystemVerilog matchesexpression is not evaluated


inferred: any


<a id="T08.14.6"></a>
## 08.14.6 · Its editions are data

Pattern matching arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions, 1800-2005 to 1800-2023,
and none of Verilog's, each with every one of its kinds
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
const singleton = store.Instances.Expressions.SystemVerilog.Patterns;
const kinds = new Map((store.registered(Patterns.union).branches as any[]).map((branch) => [branch.schema, branch.name]));
for (const [designation, edition] of Standards.standards(singleton)) {
  console.log(designation.padEnd(19), Standards.tags(edition)[1]!.padEnd(19), Standards.kinds(edition).length, "kinds");
}
console.log("each:", Standards.kinds(Standards.find(singleton, "SystemVerilog-2005")!).map((k) => kinds.get(k)).join(", "));
console.log("Verilog-2005:", Standards.find(singleton, "Verilog-2005") ? "covered" : "not covered");

IEEE Std 1800-2005  SystemVerilog-2005  6 kinds


IEEE Std 1800-2009  SystemVerilog-2009  6 kinds


IEEE Std 1800-2012  SystemVerilog-2012  6 kinds


IEEE Std 1800-2017  SystemVerilog-2017  6 kinds


IEEE Std 1800-2023  SystemVerilog-2023  6 kinds


each: variablepattern, wildcardpattern, taggedpattern, structurepattern, patternmember, parenthesizedpattern


Verilog-2005: not covered


## What you learned

- **A pattern is a term of `SystemVerilog Patterns`**: a variable, a wildcard, a tagged, structure or parenthesized
  pattern, nesting; a constant pattern is an expression, which its `Term` lists.
- **A pattern binds its variables, through what holds it**: a `matches` binds them in its conditional's consequence and
  in the conditions after it, and nowhere else.
- **It is validated as any term**: a structure pattern in order or by name, not both, and its constants' names bound.
- **It is not simplified**: each kind, and the `matches` that holds it, left in the residual, named; its editions are IEEE 1800's.

Next, sequences and properties: [SystemVerilog Assertions](08_15_SystemVerilog_Assertions.ipynb).

<!-- nav -->
[← 8.13 · SystemVerilog Types](08_13_SystemVerilog_Types.ipynb) · [Home](../../README.md) · [8.15 · SystemVerilog Assertions →](08_15_SystemVerilog_Assertions.ipynb)